# CardioBench on Kaggle — PanEcho and EchoPrime on CardiacNet (ASD / PAH)

Does the EchoNet-Dynamic result (attention / transformer heads beat mean pooling) carry over to
classification? Each video is embedded once (`--sampling both --max_frames 32`), then
`scripts/cardiacnet_temporal.py` compares 4 frame views x 4 heads on PanEcho's frame encoder, and linear probes on PanEcho / EchoPrime video embeddings (3 clip views), with the 42 `PHI*` controls
removed from train, val and test. Inputs: `xiaoweixumedicalai/abnormcardiacechovideos`; GPU + Internet on.

In [ ]:
REPO_URL = "https://github.com/emirrcodes/CardioBench"
BRANCH = "main"
MODELS = ["panecho_frames"]
VIDEO_MODELS = ["panecho", "echoprime"]
DATA_ROOT = "/kaggle/input"
OUT_DIR = "/kaggle/working/cf"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/CardioBench && git clone -q --depth 1 -b {BRANCH} {REPO_URL} /kaggle/working/CardioBench
!pip install -q "open_clip_torch>=2.31" nibabel
%cd /kaggle/working/CardioBench
!git log --oneline -1

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/cardiacnet_temporal.py", "--data_root", DATA_ROOT,
                "--out_dir", OUT_DIR, "--models", *MODELS, "--video_models", *VIDEO_MODELS], check=True)

In [ ]:
import pandas as pd
from IPython.display import display
s = pd.read_csv(f"{OUT_DIR}/summary.csv")
display(s.pivot_table(index=["task", "model"], columns="config", values="AUROC").round(3))
for task in ["ASD", "PAH"]:
    f = f"{OUT_DIR}/probe/{task}/paired_comparisons.csv"
    if __import__("os").path.exists(f):
        pc = pd.read_csv(f)
        print(task); display(pc[["model", "effect", "config", "vs", "dAUROC", "dAUROC_lo", "dAUROC_hi", "p_holm"]].round(3))

In [ ]:
# One tar for the embeddings (keeps the Output listing small), then a results zip.
!cd {OUT_DIR} && tar -cf embeddings.tar embeddings video_embeddings && rm -rf embeddings video_embeddings && ls -lh embeddings.tar
!cd {OUT_DIR} && zip -qr /kaggle/working/cf_results.zip summary.csv probe splits
!ls -lh /kaggle/working/cf_results.zip